### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from dotenv import load_dotenv

load_dotenv('../../.env')

True

### RAG pipeline


In [4]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [5]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")   

In [ ]:
qdrant_client = QdrantClient(url='http://localhost:6333')

def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text,
    )
    
    return response.data[0].embedding

def retrieve_data(query, collection_name='amazon-items-collection-01', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }

def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context

def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt

def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_model=RAGGenerationResponse,
        reasoning={"effort": "none"}
    )

    return response

def rag_pipeline(question, topk=5):
    retrieved_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieved_context)
    prompt = build_prompt(question, formatted_context)
    answer = generate_answer(prompt)
    
    final_answer = {
        'data_object': answer,
        'answer': answer.answer,
        'question': question,
        'retrieved_context_ids': retrieved_context['retrieved_context_ids'],
        'retrieved_context': retrieved_context['retrieved_context_texts'],
    }

    return final_answer

In [8]:
output = rag_pipeline("Suggest me a laptop", 3)

In [9]:
output

{'data_object': RAGGenerationResponse(answer='Based on the available products, I suggest the HP 15.6" HD Business Laptop (Product ID: B0C9ZWCZ99). It has a 6-core AMD Ryzen 5 5500U (up to 4.0GHz), 8GB RAM, 256GB SSD, a 15.6" HD micro-edge display, WiFi, USB-A and USB-C ports plus HDMI, and comes with Windows 11 Home.'),
 'answer': 'Based on the available products, I suggest the HP 15.6" HD Business Laptop (Product ID: B0C9ZWCZ99). It has a 6-core AMD Ryzen 5 5500U (up to 4.0GHz), 8GB RAM, 256GB SSD, a 15.6" HD micro-edge display, WiFi, USB-A and USB-C ports plus HDMI, and comes with Windows 11 Home.',
 'question': 'Suggest me a laptop',
 'retrieved_context_ids': ['B0C9ZWCZ99', 'B099N9F3FP', 'B09WCT9S1R'],
 'retrieved_context': ['HP 15.6" HD Busienss Laptop Newest, 6-core AMD Ryzen 5 5500U(up to 4.0GHz), 8GB RAM, 256GB SSD, USB-A&C, WiFi, Fast Charge, Windows 11 + GM Accessory 【15.6" HD micro-edge Display】Revolutionize your display and see more of what you love with this slim bezel desi

### RAG Pipeline with grounding context

In [10]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="The ID of the item used to answer the question")
    description: str = Field(description="The description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [19]:
qdrant_client = QdrantClient(url='http://localhost:6333')

def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text,
    )
    
    return response.data[0].embedding

def retrieve_data(query, collection_name='amazon-items-collection-01', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }

def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context

def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - If you are describing multiple products, list them out as a list.

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt

def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_model=RAGGenerationResponse,
        reasoning={"effort": "none"}
    )

    return response

def rag_pipeline(question, topk=5):
    retrieved_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieved_context)
    prompt = build_prompt(question, formatted_context)
    answer = generate_answer(prompt)
    
    final_answer = {
        'data_object': answer,
        'answer': answer.answer,
        'references': answer.references,
        'question': question,
        'retrieved_context_ids': retrieved_context['retrieved_context_ids'],
        'retrieved_context': retrieved_context['retrieved_context_texts'],
    }

    return final_answer

In [20]:
output = rag_pipeline("Do you have earbuds with noise cancellation?", 10)

In [27]:
output

{'data_object': RAGGenerationResponse(answer='Yes. You have noise-cancelling earbuds:\n\n- pamu Wireless Earbuds (Product ID: B09TFM1SFQ) — Active Noise Cancelling with 4 built-in microphones and 99% ambient noise reduction.\n\n- MUSICOZY Bluetooth 5.3 Headband Headphones (Product ID: B0CFHWF326) — ENC environmental noise reduction for clearer calls (note: this is a headband style, not earbuds).', references=[RAGUsedContext(id='B09TFM1SFQ', description='pamu Wireless Earbuds Active Noise Cancelling earbuds with 4 built-in microphones and ANC/99% ambient noise reduction.'), RAGUsedContext(id='B0CFHWF326', description='MUSICOZY Bluetooth 5.3 sports headband with built-in ENC mic for environmental noise reduction (headband style).')]),
 'answer': 'Yes. You have noise-cancelling earbuds:\n\n- pamu Wireless Earbuds (Product ID: B09TFM1SFQ) — Active Noise Cancelling with 4 built-in microphones and 99% ambient noise reduction.\n\n- MUSICOZY Bluetooth 5.3 Headband Headphones (Product ID: B0CFH

In [28]:
print(output['references'])

[RAGUsedContext(id='B09TFM1SFQ', description='pamu Wireless Earbuds Active Noise Cancelling earbuds with 4 built-in microphones and ANC/99% ambient noise reduction.'), RAGUsedContext(id='B0CFHWF326', description='MUSICOZY Bluetooth 5.3 sports headband with built-in ENC mic for environmental noise reduction (headband style).')]


In [29]:
print(output['answer'])

Yes. You have noise-cancelling earbuds:

- pamu Wireless Earbuds (Product ID: B09TFM1SFQ) — Active Noise Cancelling with 4 built-in microphones and 99% ambient noise reduction.

- MUSICOZY Bluetooth 5.3 Headband Headphones (Product ID: B0CFHWF326) — ENC environmental noise reduction for clearer calls (note: this is a headband style, not earbuds).
